In [13]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [14]:
import os
from dotenv import load_dotenv
from src.paths import PARENT_DIR

load_dotenv(override=True)

HOPSWORKS_PROJECT_NAME = os.environ['HOPSWORKS_PROJECT_NAME']
HOPSWORKS_API_KEY = os.environ['HOPSWORKS_API_KEY']

In [15]:
print(f'project name is: {HOPSWORKS_PROJECT_NAME}')
print(f'the api key is: {HOPSWORKS_API_KEY[:5]}')

project name is: taxi_demand_prediction
the api key is: 657pq


In [16]:
from datetime import datetime
import pandas as pd
from src.data import load_row_data

from_year = 2025
to_year = datetime.now().year
print(f'Downloading row data from {from_year} to {to_year}')

rides = pd.DataFrame()
for year in range(from_year, to_year+1):

    # Download the data for the whole year
    rides_one_year = load_row_data(year)

    # appends raw
    rides = pd.concat([rides, rides_one_year])


file 2025-01 was already in local storage
file 2025-02 was already in local storage
file 2025-03 was already in local storage
file 2025-04 was already in local storage
file 2025-05 was already in local storage
file 2025-06 was already in local storage
file 2025-07 was already in local storage
file 2025-08 was already in local storage
file 2025-09 was already in local storage
file 2025-10 was already in local storage
file 2025-11 was already in local storage
file 2025-12 was already in local storage
file 2026-01 was already in local storage
file 2026-02 was already in local storage
file 2026-03 was already in local storage
file 2026-04 was already in local storage
2026-05 is not available
2026-06 is not available
2026-07 is not available
2026-08 is not available
2026-09 is not available
2026-10 is not available
2026-11 is not available
2026-12 is not available


In [17]:
print(f'{len(rides)}')

63630780


In [18]:
from src.data import tranfrom_row_data_into_ts_data

ts_data = tranfrom_row_data_into_ts_data(rides)

100%|██████████| 262/262 [00:06<00:00, 43.25it/s]


In [19]:
import hopsworks

In [20]:
project = hopsworks.login(
    project=HOPSWORKS_PROJECT_NAME,
    api_key_value=HOPSWORKS_API_KEY,
    cert_folder="./hopsworks-certs",
)

2026-06-24 10:42:04,048 INFO: Closing external client and cleaning up certificates.
2026-06-24 10:42:04,052 INFO: Connection closed.
2026-06-24 10:42:04,053 INFO: Initializing external client
2026-06-24 10:42:04,053 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-06-24 10:42:05,648 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/34960


In [21]:
feature_store = project.get_feature_store()

In [22]:
FEATURE_GROUP_NAME = "time_series_hourly_feature_group"
FEATURE_GROUP_VERSION = 1

In [23]:
feature_group = feature_store.get_or_create_feature_group(
    name=FEATURE_GROUP_NAME,
    version=FEATURE_GROUP_VERSION,
    description="Time-series data at hourly frequency",
    primary_key=["pickup_location_id", "pickup_hour"],
    event_time="pickup_hour",
    time_travel_format="HUDI",
)

In [24]:
feature_group.insert(
    ts_data,
    write_options={"wait_for_job": True}
)

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/34960/fs/22624/fg/43270


Uploading Dataframe: 100.00% |██████████| Rows 3049680/3049680 | Elapsed Time: 19:59 | Remaining Time: 00:00


Launching job: time_series_hourly_feature_group_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/34960/jobs/named/time_series_hourly_feature_group_1_offline_fg_materialization/executions


2026-06-24 11:02:22,123 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-06-24 11:02:25,301 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-06-24 11:03:32,118 INFO: Waiting for execution to finish. Current state: QUEUED. Final status: UNDEFINED
2026-06-24 11:03:35,292 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-06-24 11:06:17,378 INFO: Waiting for execution to finish. Current state: FINISHED. Final status: SUCCEEDED
2026-06-24 11:06:17,970 INFO: Waiting for log aggregation to finish.
2026-06-24 11:06:17,978 INFO: Execution finished successfully.


(Job('time_series_hourly_feature_group_1_offline_fg_materialization', 'SPARK'),
 None)

In [25]:
feature_group.read().head()

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (31.10s) 


,pickup_hour,rides,pickup_location_id
0,2025-03-14 22:00:00+00:00,0,6
1,2025-12-08 02:00:00+00:00,0,123
2,2026-02-15 04:00:00+00:00,7,228
3,2025-11-20 13:00:00+00:00,3,188
4,2025-04-17 11:00:00+00:00,5,37
